In [29]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [31]:
# 1. Load the document from the file
# doc = TextLoader(file_path='diabetes_factsheet.txt')
# dia = doc.load()
# dia

dir_loader = DirectoryLoader('./metadata', loader_cls=TextLoader)
documents = dir_loader.load()

In [ ]:
# 2. Break this doc into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 50, chunk_overlap = 10)
chunks = splitter.split_documents(documents)
chunks

In [33]:
# 3. Perform embedding of these chunks
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')
chunk_lists = [chunk.page_content for chunk in chunks]
embedded_chunks = embeddings.embed_documents(chunk_lists)
len(embedded_chunks)

91

In [34]:
# 4. Now Lets store this embedded data to the
from langchain_community.vectorstores import FAISS

vector_storage = FAISS.from_documents(chunks, embeddings)

# 5. Test

question = 'what is diabeties'


res = vector_storage.similarity_search(question, k= 4)


for doc in res:
    print(doc.page_content)

retriever = vector_storage.as_retriever(search_kwargs={'k': 3})

diabetes.
Diabetes is a chronic disease characterized by
Diabetes Factsheet
doctor or diabetes nurse.


In [36]:
# 6. Finally feed this to an LLM and then fetch the response:

from langchain_openai import OpenAI
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate


model = OpenAI(model= 'gpt-4o-mini')

prompt = '''You are a health care expert 
{context}
Question: {input}
Answer here: 
'''


prompt_template= PromptTemplate(template= prompt, input_variables=['context', 'input'])
chain = create_stuff_documents_chain(model, prompt_template)

retriever_chain = create_retrieval_chain(retriever, combine_docs_chain= chain)

# result = retriever_chain.invoke({'input': 'What is diabeties ?'})
# print(result['answer'])

paracetamol = retriever_chain.invoke({'input': 'What are the side effects of Paracetamol? '})
print(paracetamol['answer'])

Paracetamol is generally well tolerated, and serious side effects are rare when used as directed. However, some minor side effects may occur, including nausea, headache, rash, or allergic reactions in sensitive individuals. Overdose can lead to severe liver damage, so it is crucial to adhere to the recommended dosage. If you experience any unusual symptoms or signs of an allergic reaction, seek medical attention promptly. Always consult a healthcare professional if you have any concerns regarding the use of Paracetamol.


In [37]:

policy = retriever_chain.invoke({'input': 'What is the discharge policy of ICU patients ?'})
print(policy['answer'])

The discharge policy for ICU patients involves a coordinated effort from a multidisciplinary team, which typically includes intensivists, nurses, pharmacists, social workers, and other relevant healthcare professionals. The team assesses the patient's clinical stability, readiness for transfer, and appropriate level of care needed post-discharge. 

Discharge planning begins upon admission, with regular assessments and communication among team members to ensure a smooth transition. Key factors considered during the discharge process include the patient's medical condition, potential complications, need for rehabilitation services, and family support. 

Once the team agrees on the discharge plan, it is communicated to the patient and their family, ensuring they understand post-discharge care instructions and follow-up appointments. Discharge from the ICU is typically coordinated around noon, allowing for a systematic transfer to a lower level of care, such as a step-down unit or general 

In [39]:
policy = retriever_chain.invoke({'input': 'What is the discharge time for our hospital ?'})
print(policy['answer'])

The discharge time for our hospital is 12:00 noon for general wards. Discharge from the ICU is coordinated with the Hospital Discharge Policy.


In [38]:
diabetes = retriever_chain.invoke({'input': '''How is diabetes managed as per our hospital's guideline'''})
print(diabetes['answer'])

Diabetes management in our hospital follows a comprehensive approach that includes regular blood glucose monitoring, individualized meal planning, physical activity recommendations, and medication management. Patients are encouraged to self-monitor their blood glucose levels and maintain a log to share with their healthcare team. Insulin administration guidelines are provided based on the patient's specific needs, which are determined through blood glucose readings, lifestyle factors, and physician recommendations. Education on recognizing and managing hypoglycemia and hyperglycemia is also an essential component of the management plan, ensuring patients are equipped to make informed decisions about their health. Regular follow-ups with healthcare providers are scheduled to assess the effectiveness of the management plan and make necessary adjustments. Additionally, we emphasize the importance of emotional support and resources to help patients cope with the psychological aspects of li